***Problem - 1: Perform a classification task with knn from scratch.***

***1. Load the Dataset:***

• Read the dataset into a pandas DataFrame.

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


df = pd.read_csv('drive/MyDrive/Dataset/diabetes_.csv')

• Display the first few rows and perform exploratory data analysis (EDA) to understand the dataset
(e.g., check data types, missing values, summary statistics)

In [39]:
print(df.head())

print("\nCheck data types:-")
print(df.dtypes)

print("\nCheck missing values:-")
print(df.isnull().sum)

print("\nSummary statistics:-")
print(df.describe())

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Check data types:-
Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           i

***2. Handle Missing Data:***

***• Handle any missing values appropriately, either by dropping or imputing them based on the data.***

In [40]:
import numpy as np

print("Identify Columns with Invalid Zero Values:")
columns_with_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print((df[columns_with_zero] == 0).sum())

print("\nReplace Invalid Zero Values with NaN:")
df[columns_with_zero] = df[columns_with_zero].replace(0, np.nan)
print("Zero values replaced with NaN.")

print("\nCheck Missing Values After Replacement:")
print(df.isnull().sum())

print("\nImpute Missing Values with Median:")
# Median is preferred as it is robust to outliers.
for col in columns_with_zero:
    df[col].fillna(df[col].median(), inplace=True)
print("Missing values imputed using median.")

print("\nFinal Check (Missing Values After Imputation):")
print(df.isnull().sum())


Identify Columns with Invalid Zero Values:
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64

Replace Invalid Zero Values with NaN:
Zero values replaced with NaN.

Check Missing Values After Replacement:
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64

Impute Missing Values with Median:
Missing values imputed using median.

Final Check (Missing Values After Imputation):
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


/tmp/ipython-input-465197888.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


***3. Feature Engineering:***

***• Separate the feature matrix (X) and target variable (y).***

In [41]:
# Separate features and target
X = df.drop('Outcome', axis=1).values
y = df['Outcome'].values

print("Feature Matrix Shape (X):", X.shape)
print("Target Vector Shape (y):", y.shape)


Feature Matrix Shape (X): (768, 8)
Target Vector Shape (y): (768,)


***• Perform a train - test split from scratch using a 70% − 30% ratio.***

In [42]:
# Set random seed for reproducibility
np.random.seed(42)

# Total number of samples
n_samples = X.shape[0]

# Create shuffled indices
indices = np.random.permutation(n_samples)

# Compute split index (70%)
split_index = int(0.7 * n_samples)

# Split indices
train_indices = indices[:split_index]
test_indices = indices[split_index:]

# Create train-test sets
X_train = X[train_indices]
X_test = X[test_indices]

y_train = y[train_indices]
y_test = y[test_indices]

print("\nTrain-Test Split Completed")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)



Train-Test Split Completed
X_train shape: (537, 8)
X_test shape: (231, 8)
y_train shape: (537,)
y_test shape: (231,)


***4. Implement KNN:***

***• Build the KNN algorithm from scratch (no libraries like sickit-learn for KNN).***

**1. Euclidean Distance Function**

In [43]:
def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))


***2. Write functions for:***

***– Predicting the class for a single query.***

In [44]:
def knn_predict_single(X_train, y_train, query_point, k):
    distances = []

    # Compute distance from query point to all training points
    for i in range(len(X_train)):
        dist = euclidean_distance(query_point, X_train[i])
        distances.append((dist, y_train[i]))

    # Sort by distance
    distances.sort(key=lambda x: x[0])

    # Get k nearest neighbors
    k_nearest_labels = [label for (_, label) in distances[:k]]

    # Majority voting
    predicted_class = max(set(k_nearest_labels), key=k_nearest_labels.count)

    return predicted_class


***– Predicting classes for all test samples.***

In [45]:
def knn_predict(X_train, y_train, X_test, k):
    predictions = []

    for query_point in X_test:
        pred = knn_predict_single(X_train, y_train, query_point, k)
        predictions.append(pred)

    return np.array(predictions)


***3. Evaluate the performance using accuracy.***

In [46]:
def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)


***4. Run KNN and Evaluate Performance***

In [48]:
# Choose value of k
k = 5

# Make predictions
y_pred = knn_predict(X_train, y_train, X_test, k)

# Calculate accuracy
acc = accuracy(y_test, y_pred)

print("K value:", k)
print("Accuracy:", acc)

print("Total Test Samples:", len(y_test))
print("Correct Predictions:", np.sum(y_test == y_pred))
print("Incorrect Predictions:", np.sum(y_test != y_pred))



K value: 5
Accuracy: 0.7229437229437229
Total Test Samples: 231
Correct Predictions: 167
Incorrect Predictions: 64


***Problem - 2 - Experimentation:***

**`*1. Repeat the Classification Task:*`**

***• Scale the Feature matrix X.***

In [49]:
def standard_scaler(X):
    return (X - X.mean(axis=0)) / X.std(axis=0)

***• Use the scaled data for training and testing the kNN Classifier.***